# 04 · Climate drivers and CH2025 projections of larch autumn phenology

One method throughout: **linear regression with zone fixed effects**, `doy ~ climate features + C(zone)`. The zone terms absorb the fixed differences between zones (elevation, aspect), so the climate slopes come from year-to-year variation within zones.

1. **Which climate factors influence larch autumn phenology, and how?**
   - 3a: the effect of each climate feature on its own (days per °C, per 100 mm, …)
   - 3b: a multiple regression with the best `MAX_FEATURES` features
2. **Do the CH2025 climate scenarios give better predictions of future yellowing?**
   - 4a: leave-one-year-out cross-validation of the regression against a mean-only model and a linear trend (as in `03`)
   - 4b: the regression projected with all CH2025 model chains and warming levels

The phenology date to analyse is selectable: `TARGETS` in `config.py`, either a chosen date of `03` (`"start"`, `"peak"`, `"end"`) or one candidate of `SEASON_METHODS` (e.g. `"Whittaker / GRVI / mid"`). Every step runs for each target.

**Climate sources**, used side by side:
- **grid:** gridded observations, projected with CH2025 DAILY-GRIDDED
- **station:** Buffalora observations, projected with CH2025 DAILY-LOCAL

A source whose files are missing is skipped.

- **Reads:**
  - `results/<AOI>/season_selected.csv` and `season_dates.csv` (from `03`)
  - `cache/climate/<AOI>/*_zones.nc` (from `02`)
- **Writes:** `results/<AOI>/climate/`

**Sample size.** Climate varies mainly from year to year, and all zones share the same year, so the effective sample is about the number of years (≈ 10), not the number of zone-years. CIs are therefore clustered by year and the cross-validation leaves out whole years. Treat effects as indicative.

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from pyproj import Transformer
from scipy.stats import t as t_dist

import importlib
import config, phenology_utils
importlib.reload(config); importlib.reload(phenology_utils)  # pick up edits to config.py without a kernel restart
from config import *
from phenology_utils import load_zone_table

CLIMATE_FILES = {   # source: (observations, scenarios)
    "grid": ("obs_grid_zones.nc", "ch2025_grid_zones.nc"),
    "station": ("obs_station_zones.nc", "ch2025_local_zones.nc"),
}
MAX_FEATURES = 2                                # climate features per multiple regression
REFERENCE = "ref91-20"
# Approximate time when the global warming levels are reached (IPCC AR6 WG1, central estimates)
GWL_TIMING = {"ref91-20": "1991-2020", "gwl1.5": "~2030s", "gwl2.0": "~2040s-2060s",
              "gwl2.5": "~2050s-2080s", "gwl3.0": "~2060s-2100 (high emissions only)"}

CLIMATE_OUT_DIR.mkdir(parents=True, exist_ok=True)
zone_table = load_zone_table()
ZONES = [z for z in zone_table.index if z != "all"]
ZONE_SHARE = zone_table.loc[ZONES, "n_pixels"] / zone_table.loc[ZONES, "n_pixels"].sum()   # for AOI means
DOYS = np.arange(1, 366)
print(f"AOI: {AOI_NAME} | years {FIRST_YEAR}-{LAST_YEAR} | targets: {TARGETS}")

## 1. Phenology dates (targets)

`"start"`, `"peak"` and `"end"` are the chosen dates of `03` (`SEASON_CHOICE`, complete years only); a candidate such as `"Whittaker / GRVI / mid"` is taken from `season_dates.csv`, without the preliminary years. Models use the topographic zones; `"all"` is only shown for reference.

In [ ]:
def load_targets():
    # "start" / "peak" / "end": the chosen dates of 03; "<method> / <index> / <metric>": one candidate of 03
    selected = pd.read_csv(OUT_DIR / "season_selected.csv")
    dates = pd.read_csv(OUT_DIR / "season_dates.csv")
    dates = dates[~dates.preliminary]
    parts = []
    for target in TARGETS:
        if target in ("start", "peak", "end"):
            d = selected.rename(columns={target: "doy"})
        else:
            key = [part.strip() for part in target.split("/")]
            d = (dates[(dates.method == key[0]) & (dates["index"] == key[1]) & (dates.metric == key[2])]
                 if len(key) == 3 else dates.iloc[:0])
            if d.empty:
                options = sorted(set(dates.method + " / " + dates["index"] + " / " + dates.metric))
                raise ValueError(f"TARGETS: {target!r} is not in season_dates.csv. Options: start, peak, end, {options}")
        parts.append(d[["zone", "year", "doy"]].dropna().assign(target=target))
    out = pd.concat(parts, ignore_index=True)
    return out[out.year.between(FIRST_YEAR, LAST_YEAR)]


targets = load_targets()
display(targets[targets.zone != "all"].pivot_table(index="target", columns="year", values="doy", aggfunc="size")
        .fillna(0).astype(int).rename_axis(columns="zones per year"))

fig, axes = plt.subplots(1, len(TARGETS), figsize=(4 * len(TARGETS), 3.5), sharey=True, squeeze=False)
for ax, target in zip(axes[0], TARGETS):
    d = targets[targets.target == target]
    for zone, g in d[d.zone != "all"].groupby("zone"):
        ax.plot(g.year, g.doy, "-", lw=0.8, alpha=0.5)
    a = d[d.zone == "all"]
    ax.plot(a.year, a.doy, "ko-", lw=2, label="whole AOI")
    ax.set_title(target)
    ax.set_xlabel("year")
    ax.grid(alpha=0.3)
axes[0, 0].set_ylabel("day of year")
axes[0, 0].legend()
plt.suptitle("Phenology dates per zone (thin) and for the whole AOI")
plt.tight_layout()
plt.show()

## 2. Climate data and features per zone and year

The daily zone series of `02` are arranged as (zone, year, day of year) with 365 days; 29 February is dropped. Model years of CH2025 are numbered 1–30 per warming level.

The **features** are seasonal summaries of each year. They are computed the same way for observations and scenarios, and only from the variables a source has. The windows end by mid October so that they precede most transition dates. Snow is not in the scenarios; winter precipitation serves as a snowpack proxy.

In [ ]:
def to_matrix(ds):
    # (..., time) -> (..., year, doy); 29 Feb dropped so every year has 365 days
    leap_day = (ds.month == 2) & (ds.doy == 60)
    ds = ds.isel(time=np.flatnonzero(~leap_day.values)).drop_vars("month")
    return ds.set_index(time=["year", "doy"]).unstack("time").reindex(doy=DOYS)


climate = {}
for source, (obs_file, scen_file) in CLIMATE_FILES.items():
    if not (CLIMATE_DIR / obs_file).exists():
        print(f"{source}: {obs_file} missing (run 02_download_climate), source skipped")
        continue
    obs = to_matrix(xr.open_dataset(CLIMATE_DIR / obs_file).load())
    scen = None
    if (CLIMATE_DIR / scen_file).exists():
        scen = to_matrix(xr.open_dataset(CLIMATE_DIR / scen_file).load())
    else:
        print(f"{source}: {scen_file} missing, no projections for this source")
    climate[source] = {"obs": obs.sel(zone=ZONES), "scen": None if scen is None else scen.sel(zone=ZONES)}
    print(f"{source}: observations {list(obs.data_vars)}, years {int(obs.year.min())}-{int(obs.year.max())}"
          + ("" if scen is None else f" | scenarios {dict(scen.sizes)}"))
if not climate:
    raise FileNotFoundError(f"No climate data in {CLIMATE_DIR}; run 02_download_climate first")

cells = pd.read_csv(CLIMATE_DIR / "zone_cells.csv")
_, LATITUDE = Transformer.from_crs("EPSG:2056", "EPSG:4326", always_xy=True).transform(cells.E.mean(), cells.N.mean())

In [ ]:
def extraterrestrial_radiation(lat, doy):
    # FAO-56 eq. 21, MJ m-2 day-1
    phi = np.radians(lat)
    dr = 1 + 0.033 * np.cos(2 * np.pi * doy / 365)
    delta = 0.409 * np.sin(2 * np.pi * doy / 365 - 1.39)
    ws = np.arccos(-np.tan(phi) * np.tan(delta))
    return 24 * 60 / np.pi * 0.0820 * dr * (ws * np.sin(phi) * np.sin(delta) + np.cos(phi) * np.cos(delta) * np.sin(ws))


RA = xr.DataArray(extraterrestrial_radiation(LATITUDE, DOYS), dims="doy", coords={"doy": DOYS})


def season(da, first, last, how="mean"):
    # Mean or sum over DOY first..last of every year; NaN if a day is missing
    win = da.sel(doy=slice(first, last))
    return win.mean("doy", skipna=False) if how == "mean" else win.sum("doy", skipna=False)


def winter_sum(da):
    # 1 Nov of the previous year - 30 Apr
    return season(da, 305, 365, "sum").shift(year=1) + season(da, 1, 120, "sum")


def pet_hargreaves(ds):
    # mm/day (Hargreaves & Samani 1985)
    return 0.0023 * 0.408 * RA * (ds.tas + 17.8) * np.sqrt((ds.tasmax - ds.tasmin).clip(min=0))


def vpd(ds):
    # Vapour pressure deficit (hPa) from mean temperature and relative humidity
    return 6.112 * np.exp(17.62 * ds.tas / (243.12 + ds.tas)) * (1 - ds.hurs / 100)


# name: (description, unit, required variables, function of the daily dataset)
FEATURES = {
    "T_spring": ("mean temperature Mar-May", "°C", ["tas"], lambda d: season(d.tas, 60, 151)),
    "GDD5_JanJul": ("growing degree-days above 5 °C, Jan-Jul", "°C d", ["tas"],
                    lambda d: season((d.tas - 5).clip(min=0), 1, 212, "sum")),
    "T_summer": ("mean temperature Jun-Aug", "°C", ["tas"], lambda d: season(d.tas, 152, 243)),
    "T_Sep": ("mean temperature Sep", "°C", ["tas"], lambda d: season(d.tas, 244, 273)),
    "T_Oct1": ("mean temperature 1-15 Oct", "°C", ["tas"], lambda d: season(d.tas, 274, 288)),
    "Tmin_Sep": ("mean minimum temperature Sep", "°C", ["tasmin"], lambda d: season(d.tasmin, 244, 273)),
    "frost_days": ("days with Tmin < 0 °C, 15 Aug-15 Oct", "days", ["tasmin"],
                   lambda d: season((d.tasmin < 0).where(d.tasmin.notnull()), 227, 288, "sum")),
    "CDD10_AugSep": ("cold degree-days below 10 °C, Aug-Sep", "°C d", ["tas"],
                     lambda d: season((10 - d.tas).clip(min=0), 213, 273, "sum")),
    "P_winter": ("precipitation Nov-Apr (snowpack proxy)", "mm", ["pr"], lambda d: winter_sum(d.pr)),
    "P_summer": ("precipitation Jun-Aug", "mm", ["pr"], lambda d: season(d.pr, 152, 243, "sum")),
    "P_AugSep": ("precipitation Aug-Sep", "mm", ["pr"], lambda d: season(d.pr, 213, 273, "sum")),
    "CWB_summer": ("climatic water balance P - PET (Hargreaves), Jun-Sep", "mm", ["pr", "tas", "tasmin", "tasmax"],
                   lambda d: season(d.pr - pet_hargreaves(d), 152, 273, "sum")),
    "sunshine_AugSep": ("relative sunshine duration Aug-Sep", "%", ["srel"], lambda d: season(d.srel, 213, 273)),
    "radiation_AugSep": ("global radiation Aug-Sep", "W/m²", ["rsds"], lambda d: season(d.rsds, 213, 273)),
    "VPD_summer": ("vapour pressure deficit Jul-Sep", "hPa", ["tas", "hurs"], lambda d: season(vpd(d), 182, 273)),
    "ET0_summer": ("FAO reference evaporation Jun-Sep", "mm", ["et0"], lambda d: season(d.et0, 152, 273, "sum")),
}


def compute_features(ds):
    return xr.Dataset({name: f(ds) for name, (_, _, needs, f) in FEATURES.items() if set(needs) <= set(ds.data_vars)})


info_rows, obs_features = [], {}
for source, c in climate.items():
    feats = compute_features(c["obs"])
    obs_features[source] = feats.to_dataframe().reset_index()
    c["features_scen"] = None if c["scen"] is None else compute_features(c["scen"])
    for name in feats.data_vars:
        info_rows.append({"source": source, "feature": name, "description": FEATURES[name][0],
                          "unit": FEATURES[name][1],
                          "in_scenarios": c["features_scen"] is not None and name in c["features_scen"]})
feature_info = pd.DataFrame(info_rows)
display(feature_info.pivot_table(index=["feature", "description", "unit"], columns="source", values="in_scenarios",
                                 aggfunc="first").replace({True: "obs + scenarios", False: "obs only"}))

PLOT_SOURCE = "grid" if "grid" in climate else next(iter(climate))   # source of the partial plots


def model_data(target, source):
    # One row per zone-year: phenology date + features
    d = targets[(targets.target == target) & targets.zone.isin(ZONES)]
    return d.merge(obs_features[source], on=["zone", "year"], how="inner").reset_index(drop=True)


def candidate_features(source, d):
    # Features usable in a model: in the observations and, if the source has scenarios, in the scenarios too (the
    # models are used for projection); at least 80 % of the zone-years must have a value
    c = climate[source]
    feats = [f for f in obs_features[source].columns.drop(["zone", "year"])
             if c["features_scen"] is None or f in c["features_scen"]]
    return [f for f in feats if d[f].notna().mean() > 0.8]


## 3. Q1: Which climate factors matter, and how?

All models are OLS with zone fixed effects, `doy ~ features + C(zone)`, fitted to the zone-years. Equivalently, date and features are centred on each zone's mean and regressed on each other, so elevation differences between zones cannot pose as climate effects.

- **`slope`:** change in date per unit of the feature. CIs are clustered by year, using t quantiles with (years − 1) degrees of freedom.
- **`r2_within`:** the share of the within-zone (year-to-year) variation explained.

### 3a. Effect of each climate feature

One regression per target and feature. **`space_for_time`** is the slope across the zone means (the elevation gradient), for comparison: if it is similar to the slope over the years, warmer sites behave like warmer years; if not, other site factors play a role. It is empty for features that are the same in every zone (e.g. station precipitation).

In [ ]:
def demean(d, cols):
    return d[cols] - d.groupby("zone")[cols].transform("mean")


def fe_ols(d, feats):
    # OLS doy ~ feats + C(zone) on the zone-demeaned data; cluster-robust (by year) CIs. None if not estimable
    d = d.dropna(subset=feats + ["doy"])
    n_years = d.year.nunique()
    X, y = demean(d, feats).values, demean(d, ["doy"])["doy"].values
    if n_years < 4 or np.linalg.matrix_rank(X) < len(feats):
        return None
    xtx_inv = np.linalg.inv(X.T @ X)
    beta = xtx_inv @ X.T @ y
    resid = y - X @ beta
    score = pd.DataFrame(X * resid[:, None]).groupby(d.year.values).sum().values
    se = np.sqrt(np.diag(n_years / (n_years - 1) * xtx_inv @ (score.T @ score) @ xtx_inv))
    q = t_dist.ppf(1 - ALPHA / 2, n_years - 1)
    coef = pd.DataFrame({"slope": beta, "ci_low": beta - q * se, "ci_high": beta + q * se,
                         "p": 2 * t_dist.sf(np.abs(beta / se), n_years - 1)}, index=feats)
    return {"coef": coef, "r2_within": 1 - (resid @ resid) / (y @ y), "n_zone_years": len(d), "n_years": n_years}


effect_rows = []
for source in climate:
    for target in TARGETS:
        d = model_data(target, source)
        for feature in obs_features[source].columns.drop(["zone", "year"]):
            fit = fe_ols(d, [feature])
            if fit is None:
                continue
            zone_means = d.dropna(subset=[feature]).groupby("zone")[[feature, "doy"]].mean()
            spread = zone_means[feature].max() - zone_means[feature].min()   # 0 if the feature is the same in all zones
            space = (np.polyfit(zone_means[feature], zone_means["doy"], 1)[0]
                     if len(zone_means) >= 3 and spread > 1e-6 * max(1, zone_means[feature].abs().max()) else np.nan)
            effect_rows.append({"source": source, "target": target, "feature": feature,
                                "unit": f"days per {FEATURES[feature][1]}", **fit["coef"].iloc[0].to_dict(),
                                "r2_within": fit["r2_within"], "space_for_time": space,
                                "n_zone_years": fit["n_zone_years"], "n_years": fit["n_years"]})
climate_effects = pd.DataFrame(effect_rows)
for (source, target), g in climate_effects.groupby(["source", "target"], sort=False):
    print(f"{target} ({source} observations)")
    display(g.drop(columns=["source", "target"]).sort_values("p").set_index("feature").round(3))

In [ ]:
# Partial plots of the three strongest features per target (within-zone deviations, coloured by year)
for target in TARGETS:
    d = model_data(target, PLOT_SOURCE)
    top = climate_effects[(climate_effects.source == PLOT_SOURCE) & (climate_effects.target == target)].nsmallest(3, "p")
    if top.empty:
        continue
    fig, axes = plt.subplots(1, len(top), figsize=(4.2 * len(top), 3.6), squeeze=False)
    for ax, row in zip(axes[0], top.itertuples()):
        dd = d.dropna(subset=[row.feature])
        x, y = demean(dd, [row.feature])[row.feature], demean(dd, ["doy"])["doy"]
        sc = ax.scatter(x, y, c=dd.year, cmap="viridis", s=18)
        xx = np.array([x.min(), x.max()])
        ax.plot(xx, row.slope * xx, "k-")
        ax.set_xlabel(f"{row.feature} deviation ({FEATURES[row.feature][1]})")
        ax.set_title(f"{row.slope:+.2f} d per {FEATURES[row.feature][1]} (p={row.p:.3f})", fontsize=10)
        ax.grid(alpha=0.3)
    axes[0, 0].set_ylabel("date deviation (days)")
    fig.colorbar(sc, ax=axes, label="year")
    plt.suptitle(f"{target}: strongest climate effects ({PLOT_SOURCE} observations)")
    plt.show()

### 3b. Multiple regression

Up to `MAX_FEATURES` features are combined in one model per source and target. They are chosen by forward selection: at each step the feature that most lowers the leave-one-year-out RMSE is added, as long as it lowers it. Candidates are the features that also exist in the scenarios of the source (`candidate_features`), because this model is the one projected in 4b.

The table shows the final model, fitted on all years. The p-values do not account for the selection and are therefore optimistic; section 4a gives the honest out-of-sample test.

In [ ]:
def fe_fit(d, feats):
    # OLS with zone fixed effects as dummies, for prediction: doy ~ feats + zone
    zones = pd.get_dummies(d.zone, dtype=float)
    X = np.column_stack([d[feats].values, zones.values]) if feats else zones.values
    beta = np.linalg.lstsq(X, d.doy.values, rcond=None)[0]
    return {"features": list(feats), "slopes": beta[:len(feats)], "zones": pd.Series(beta[len(feats):], zones.columns)}


def fe_predict(model, d):
    base = d.zone.map(model["zones"]).values
    return base + (d[model["features"]].values @ model["slopes"] if model["features"] else 0)


def loyo_rmse(d, feats):
    err = []
    for year in d.year.unique():
        tr, te = d[d.year != year], d[d.year == year]
        err.append(fe_predict(fe_fit(tr, feats), te) - te.doy.values)
    err = np.concatenate(err)
    return np.sqrt(np.nanmean(err ** 2))


def select_features(d, candidates):
    chosen, best = [], loyo_rmse(d, [])
    while len(chosen) < MAX_FEATURES:
        scores = {f: loyo_rmse(d, chosen + [f]) for f in candidates if f not in chosen}
        if not scores:
            break
        f, s = min(scores.items(), key=lambda kv: kv[1])
        if s >= best:
            break
        chosen.append(f)
        best = s
    return chosen


def regression_data(target, source):
    # Zone-years of one target with all candidate features (complete rows), and the candidates
    d = model_data(target, source)
    feats = candidate_features(source, d)
    return d.dropna(subset=feats).reset_index(drop=True), feats


final_models, reg_rows = {}, []
for source in climate:
    for target in TARGETS:
        d, feats = regression_data(target, source)
        if d.year.nunique() < 4:
            print(f"{target} ({source}): fewer than 4 years, skipped")
            continue
        chosen = select_features(d, feats)
        final_models[(source, target)] = {"data": d, "model": fe_fit(d, chosen)}
        fit = fe_ols(d, chosen) if chosen else None
        if fit is None:
            reg_rows.append({"source": source, "target": target, "feature": "(none: no feature beats the zone means)"})
            continue
        for feature, c in fit["coef"].iterrows():
            reg_rows.append({"source": source, "target": target, "feature": feature,
                             "unit": f"days per {FEATURES[feature][1]}", **c.to_dict(),
                             "r2_within": fit["r2_within"], "n_zone_years": fit["n_zone_years"],
                             "n_years": fit["n_years"]})
climate_regression = pd.DataFrame(reg_rows)
display(climate_regression.set_index(["source", "target", "feature"]).round(3))

## 4. Q2: Does the climate regression predict better than the trend?

### 4a. Leave-one-year-out cross-validation

Each year is predicted by models fitted on the other years only.

| Model | How it predicts |
|---|---|
| mean | each zone's mean date |
| trend | zone means + a common linear trend in years (the baseline of `03`) |
| climate regression | zone means + up to `MAX_FEATURES` climate features (3b); the feature selection is repeated in every fold, so the score is not optimistic |

**`skill`** = 1 − RMSE / RMSE(mean). Above 0 means better than the mean-only model.

In [ ]:
def scores(pred, obs):
    err = pred - obs
    return {"rmse": np.sqrt(np.nanmean(err ** 2)), "mae": np.nanmean(np.abs(err)), "bias": np.nanmean(err),
            "r": pd.Series(pred).corr(pd.Series(obs)), "n": int(np.isfinite(err).sum())}


cv_rows, chosen_rows = [], []
for (source, target), fm in final_models.items():
    d = fm["data"]
    feats = candidate_features(source, d)
    pred = {name: np.full(len(d), np.nan) for name in ["mean", "trend", "climate regression"]}
    for year in d.year.unique():
        tr, te = d.year != year, d.year == year
        pred["mean"][te] = fe_predict(fe_fit(d[tr], []), d[te])
        pred["trend"][te] = fe_predict(fe_fit(d[tr], ["year"]), d[te])
        chosen = select_features(d[tr], feats)
        chosen_rows.append({"source": source, "target": target, "left_out": year, "features": ", ".join(chosen)})
        pred["climate regression"][te] = fe_predict(fe_fit(d[tr], chosen), d[te])
    for name, p in pred.items():
        cv_rows.append({"source": source, "target": target, "model": name, **scores(p, d.doy.values)})

cv_scores = pd.DataFrame(cv_rows)
cv_scores["skill"] = 1 - cv_scores.rmse / cv_scores.groupby(["source", "target"]).rmse.transform(
    lambda s: s[cv_scores.loc[s.index, "model"] == "mean"].iloc[0])
display(cv_scores.set_index(["source", "target", "model"]).round(2))
cv_features = pd.DataFrame(chosen_rows)
print("Features chosen in the folds (number of folds):")
display(cv_features.groupby(["source", "target"]).features.value_counts().rename("folds").to_frame())

fig, axes = plt.subplots(1, len(TARGETS), figsize=(4 * len(TARGETS), 3.5), squeeze=False, sharey=True)
for ax, target in zip(axes[0], TARGETS):
    s = cv_scores[cv_scores.target == target].pivot(index="model", columns="source", values="rmse")
    s.plot.barh(ax=ax, legend=ax is axes[0, -1])
    ax.set_title(target)
    ax.set_xlabel("RMSE (days), left-out years")
    ax.grid(alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

### 4b. Projections with CH2025

The final regression of 3b, fitted on all years, is applied to every CH2025 model chain, warming level and model year (for sources with scenarios). CH2025 is bias-adjusted to the observations, so the predicted dates are absolute dates.

Two summaries are reported per zone and warming level:
- **`doy_*`:** the 30-year mean date of each model chain. `doy_median` is the median over chains; `doy_p05` / `doy_p95` are the 5th and 95th percentiles.
- **`delta_*`:** the shift relative to `ref91-20` of the same chain. This removes the remaining model bias.

The whole AOI is the pixel-weighted mean of the zones. Check `skill` (4a) first: a regression that does not beat the mean-only model in cross-validation should not be read as a projection.

**Extrapolation:** `extrapolation` gives, per selected feature and warming level, the share of scenario zone-years outside the observed range. The regression is a straight-line guess there.

In [ ]:
def regression_projection(model, feats_scen):
    # (period, model, zone, year) predicted DOY
    zone_eff = xr.DataArray(model["zones"].reindex(ZONES).values, dims="zone", coords={"zone": ZONES})
    out = zone_eff + 0 * feats_scen[list(feats_scen.data_vars)[0]]
    for f, b in zip(model["features"], model["slopes"]):
        out = out + b * feats_scen[f]
    return out


proj_rows, extrap_rows, chain_means = [], [], []
for (source, target), fm in final_models.items():
    c = climate[source]
    if c["features_scen"] is None:
        print(f"{target} ({source}): no scenarios, no projection")
        continue
    per_chain = regression_projection(fm["model"], c["features_scen"]).mean("year")   # 30-year mean per chain
    per_chain = xr.concat([per_chain, (per_chain * xr.DataArray(ZONE_SHARE)).sum("zone", min_count=1)
                           .expand_dims(zone=["AOI"])], "zone")
    delta = per_chain - per_chain.sel(period=REFERENCE)
    obs_mean = (fm["data"].groupby("zone").doy.mean() * ZONE_SHARE).sum() / ZONE_SHARE[fm["data"].zone.unique()].sum()
    for period in per_chain.period.values:
        for zone in per_chain.zone.values:
            v = per_chain.sel(period=period, zone=zone).values
            dv = delta.sel(period=period, zone=zone).values
            if np.isfinite(v).sum() == 0:
                continue
            proj_rows.append({
                "source": source, "target": target, "period": period,
                "timing": GWL_TIMING.get(str(period), ""), "zone": zone,
                "doy_median": np.nanmedian(v), "doy_p05": np.nanpercentile(v, 5), "doy_p95": np.nanpercentile(v, 95),
                "delta_median": np.nanmedian(dv), "delta_p05": np.nanpercentile(dv, 5),
                "delta_p95": np.nanpercentile(dv, 95), "n_chains": int(np.isfinite(v).sum()),
                "observed_mean_doy": obs_mean if zone == "AOI" else fm["data"].loc[fm["data"].zone == zone, "doy"].mean(),
            })
    chain_means.append(per_chain.sel(zone="AOI").to_dataframe(name="doy").reset_index().assign(source=source, target=target))
    for f in fm["model"]["features"]:
        lo, hi = fm["data"][f].min(), fm["data"][f].max()
        sv = c["features_scen"][f]
        outside = ((sv < lo) | (sv > hi)).where(sv.notnull()).mean(["model", "zone", "year"])
        for period, share in outside.to_series().items():
            extrap_rows.append({"source": source, "target": target, "feature": f, "period": period,
                                "share_outside_observed_range": share})

projections_by_gwl = pd.DataFrame(proj_rows)
extrapolation = pd.DataFrame(extrap_rows)
if not projections_by_gwl.empty:
    display(projections_by_gwl[projections_by_gwl.zone == "AOI"].set_index(["source", "target", "period"])
            .drop(columns="zone").round(1))
if not extrapolation.empty:
    display(extrapolation.pivot_table(index=["source", "target", "feature"], columns="period",
                                      values="share_outside_observed_range").round(2))

In [ ]:
if chain_means:
    chains = pd.concat(chain_means)
    periods = [p for p in CH2025_PERIODS if p in set(chains.period)]
    sources = sorted(set(chains.source))
    skill = cv_scores[cv_scores.model == "climate regression"].set_index(["source", "target"]).skill
    for target in TARGETS:
        ct = chains[chains.target == target]
        if ct.empty:
            continue
        # Whole AOI: 30-year mean date per model chain and warming level
        fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), gridspec_kw={"width_ratios": [1.2, 1]})
        ax = axes[0]
        width = 0.8 / len(sources)
        for k, source in enumerate(sources):
            g = ct[ct.source == source]
            pos = np.arange(len(periods)) + (k - (len(sources) - 1) / 2) * width
            bp = ax.boxplot([g[g.period == p].doy.dropna() for p in periods], positions=pos, widths=width * 0.9,
                            patch_artist=True, showfliers=False)
            for patch in bp["boxes"]:
                patch.set_facecolor(plt.get_cmap("tab10")(k))
                patch.set_alpha(0.6)
            ax.plot([], [], "s", color=plt.get_cmap("tab10")(k),
                    label=f"{source} (skill {skill.get((source, target), np.nan):+.2f})")
        observed = targets[(targets.target == target) & (targets.zone == "all")].doy.mean()
        ax.axhline(observed, color="k", ls="--", label=f"observed mean {FIRST_YEAR}-{LAST_YEAR}")
        ax.set_xticks(range(len(periods)))
        ax.set_xticklabels([f"{p}\n{GWL_TIMING.get(p, '')}" for p in periods], fontsize=8)
        ax.set_ylabel("day of year (30-year mean per model chain)")
        ax.set_title(f"{target}: projected date, whole AOI")
        ax.legend(fontsize=8, loc="best")
        ax.grid(alpha=0.3, axis="y")

        # Shift per zone (elevation) and warming level, source of the partial plots
        ax = axes[1]
        p = projections_by_gwl[(projections_by_gwl.source == PLOT_SOURCE) & (projections_by_gwl.target == target)
                               & projections_by_gwl.zone.isin(ZONES)]
        for period, g in p.groupby("period"):
            g = g.join(zone_table[["elev_mean", "aspect"]], on="zone").sort_values("elev_mean")
            for aspect, ga in g.groupby("aspect"):
                ax.errorbar(ga.elev_mean, ga.delta_median,
                            yerr=[ga.delta_median - ga.delta_p05, ga.delta_p95 - ga.delta_median],
                            fmt="o-" if aspect == "N" else "s--", capsize=2, alpha=0.8, label=f"{period} {aspect}",
                            color=plt.get_cmap("plasma")(periods.index(period) / len(periods)))
        ax.axhline(0, color="k", lw=0.5)
        ax.set_xlabel("zone mean elevation (m)")
        ax.set_ylabel(f"shift vs {REFERENCE} (days)")
        ax.set_title(f"{target}: shift per zone ({PLOT_SOURCE}), median and 5-95 % of chains", fontsize=10)
        ax.legend(fontsize=7, ncol=2)
        ax.grid(alpha=0.3)
        plt.tight_layout()
        plt.show()

## 5. Export

In [ ]:
outputs = {
    "climate_features_obs.csv": pd.concat([f.assign(source=s) for s, f in obs_features.items()]),
    "feature_info.csv": feature_info,
    "climate_effects.csv": climate_effects,
    "climate_regression.csv": climate_regression,
    "cv_scores.csv": cv_scores,
    "cv_selected_features.csv": cv_features,
    "projections_by_gwl.csv": projections_by_gwl,
    "extrapolation.csv": extrapolation,
}
for name, frame in outputs.items():
    frame.to_csv(CLIMATE_OUT_DIR / name, index=False)
print(f"Results written to {CLIMATE_OUT_DIR}")